# Serving Model Selection — NYC Taxi Trip Duration

The most accurate model is a Random Forest with the tuned params (latest `optuna-tuning` run,
`uv run nyc-taxi tune`; fallback `BEST_PARAMS` in `src/train.py`). Trained on Jan 1–25 it is
several GB, which makes it slow to load and expensive to serve.

The model to **deploy** is chosen by `uv run nyc-taxi select` (`src.serving_selection run`, also part
of `uv run nyc-taxi all`). It trades a little accuracy for size, memory and speed, with a rule fixed
**before** any candidate is measured, evaluates the choice once on February 2025 and registers it in
MLflow **without an alias**.

**This notebook trains and computes nothing.** It reads the latest finished MLflow run
`serving-selection` and the three baseline runs (`baseline-overall-median`, `baseline-zone-pair-median`,
`rf-baseline`, logged by `nyc-taxi train`), and shows the rule, the comparison table, the charts,
the decision and the February result. Every candidate is compared against the baselines.

**Before running:** start MLflow (`docker compose up -d postgres mlflow`) and run the pipeline once.
Runtime: under a minute from a fresh kernel (only small files are downloaded from MLflow, not the model).

## 1. Load the selection run from MLflow

In [ ]:
import io
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import mlflow
import pandas as pd
from IPython.display import Image, display

from src import baselines as bl
from src import serving_selection as ss
from src.experiment import EXPERIMENT, TRACKING_URI
from src.train import MODEL_NAME

mlflow.set_tracking_uri(TRACKING_URI)
run = ss.latest_run(ss.PARENT_RUN_NAME, experiment=EXPERIMENT)
run_id = run.info.run_id
params, metrics, tags = run.data.params, run.data.metrics, run.data.tags
selected_name = tags["selected_candidate"]

baseline_runs, missing = bl.read_baseline_runs(EXPERIMENT)  # logged by `nyc-taxi train`
if missing:
    print(bl.missing_runs_message(missing))
baselines = bl.baseline_metrics_table(baseline_runs) if baseline_runs else pd.DataFrame()


def artifact(name):
    """Local copy of one file logged in the selection run."""
    return ss.download_run_file(run_id, name)


def show(fig):
    """Display a matplotlib Figure as a PNG."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110)
    display(Image(data=buffer.getvalue()))


print(f"MLflow: {TRACKING_URI}")
print(f"Run {run_id} ({pd.Timestamp(run.info.start_time, unit='ms'):%Y-%m-%d %H:%M} UTC)")
print(f"Selected: {selected_name}, registered as {MODEL_NAME} version {tags.get('registered_version')}")
print(f"Baseline runs found: {list(baseline_runs) or 'none'}")
print(f"Benchmark machine: {params.get('machine_cpu')}, RAM {params.get('machine_ram_gb')} GB, "
      f"{params.get('machine_os')}; Python {params.get('python_version')}, "
      f"scikit-learn {params.get('sklearn_version')}")

## 2. Selection rule (fixed before the results)

In [ ]:
print(artifact(ss.RULE_FILE).read_text(encoding="utf-8"))

## 3. Data periods

In [ ]:
periods = {
    "candidates: training": "training",
    "candidates: validation": "validation",
    "selected: final training": "final_training",
    "selected: holdout": "holdout",
}
pd.DataFrame({
    label: {"start": params.get(f"{ctx}_start"), "end": params.get(f"{ctx}_end"),
            "trips": f"{int(params.get(f'{ctx}_rows', 0)):,}"}
    for label, ctx in periods.items()
}).T

## 4. Comparison table (validation, Jan 21–25)

The three baselines come first (status `baseline`, from their own MLflow runs, scored on the same
validation days with Jan 1–20 statistics); then one row per candidate with its accuracy, serialized size
and fresh-process benchmark numbers (also logged as one child run per candidate). `rmse_vs_best_pct`
is relative to the best candidate. Baselines have no size or benchmark numbers.

In [ ]:
ranked = pd.read_csv(artifact(ss.TABLE_FILE))
best_rmse = ranked["val_rmse"].min()

if baselines.empty:
    print(bl.MISSING_RUN_HINT)
    baseline_rows = pd.DataFrame(columns=ss.TABLE_COLUMNS)
else:
    baseline_rows = baselines.reindex(columns=ss.TABLE_COLUMNS).drop(columns=["candidate", "status"])
    baseline_rows = baseline_rows.rename_axis("candidate").reset_index()
    baseline_rows["status"] = "baseline"
    baseline_rows["rmse_vs_best_pct"] = (baseline_rows["val_rmse"] / best_rmse - 1) * 100

comparison = pd.concat([baseline_rows[ss.TABLE_COLUMNS], ranked[ss.TABLE_COLUMNS]], ignore_index=True)
comparison.round(3)

Error by segment (validation): short < 5 min, long > 30 min, airport (EWR/JFK/LGA), uncommon zone pair (< 10 training trips). Baseline rows come from the baseline runs.

In [ ]:
seg_cols = [f"{s}_{m}" for s in ss.SEGMENTS for m in ("rmse", "mae")]
baseline_segments = baselines.reindex(columns=["val_rmse", "val_mae", *[f"val_{c}" for c in seg_cols]])
baseline_segments = baseline_segments.rename(columns={f"val_{c}": c for c in seg_cols})
pd.concat([baseline_segments, ranked.set_index("candidate")[["val_rmse", "val_mae", *seg_cols]]]).round(3)

## 5. Charts

Green = selected, blue = eligible, grey = rejected. Red dashed line: best RMSE + tolerance; orange line:
size budget; black lines: the baselines (a baseline far above every candidate is listed in the legend
instead, so the candidates stay readable). Drawn from the logged table and the baseline runs.

In [ ]:
policy = ss.SelectionPolicy(rmse_tolerance=float(params["rmse_tolerance"]),
                            max_size_mb=float(params["max_size_mb"]))
baseline_rmse = baselines["val_rmse"].dropna().to_dict() if "val_rmse" in baselines else {}
show(ss.plot_tradeoff(ranked, "size_mb", "serialized size (MB, skops)", "Accuracy vs size", policy,
                      size_budget=True, baselines=baseline_rmse))
show(ss.plot_tradeoff(ranked, "latency_p95_ms", "single-request p95 latency (ms)", "Accuracy vs latency",
                      policy, baselines=baseline_rmse))

## 6. Decision

Every candidate with its status and the exact reason.

In [ ]:
with pd.option_context("display.max_colwidth", 200):
    display(ranked[["candidate", "status", "reason"]].set_index("candidate"))

by_name = ranked.set_index("candidate")
selected, ref = by_name.loc[selected_name], by_name.loc["tuned-200"]
print(f"Selected: {selected_name}  val RMSE {selected['val_rmse']:.4f} "
      f"({selected['rmse_vs_best_pct']:+.2f}% vs best), size {selected['size_mb']:,.0f} MB, "
      f"peak RAM {selected['peak_ram_mb']:,.0f} MB, p95 {selected['latency_p95_ms']:.2f} ms")
print(f"vs reference tuned-200: size {1 - selected['size_mb'] / ref['size_mb']:.1%} smaller, "
      f"peak RAM {1 - selected['peak_ram_mb'] / ref['peak_ram_mb']:.1%} lower, "
      f"load {ref['load_s'] / selected['load_s']:.1f}x faster, "
      f"RMSE {selected['val_rmse'] - ref['val_rmse']:+.4f} min")
for label, rmse in baseline_rmse.items():
    print(f"vs {label}: RMSE {selected['val_rmse'] - rmse:+.4f} min ({selected['val_rmse'] / rmse - 1:+.1%})")
print(f"Repeatability (refit from scratch = measured model): {tags.get('repeatable')}, "
      f"max abs difference {metrics.get('repeat_max_abs_diff', float('nan')):.2e} min")

## 7. February 2025 holdout (evaluated once)

Only the selected configuration was retrained on all of January and evaluated once on February, an
untouched holdout. The zone-pair median uses the same January trips.

In [ ]:
def holdout_row(prefix):
    keys = ["rmse", "mae", *[f"{s}_{m}" for s in ss.SEGMENTS for m in ("rmse", "mae")]]
    return {k: metrics.get(f"{prefix}{k}", float("nan")) for k in keys}


holdout = pd.DataFrame({
    f"{selected_name} (Jan 1-31)": holdout_row("holdout_"),
    "zone-pair median (Jan 1-31)": holdout_row("holdout_zone_pair_median_"),
}).T
display(holdout.round(4))
print("Segment sizes (February):", {s: int(metrics.get(f"holdout_{s}_trips", 0)) for s in ss.SEGMENTS})
print(f"Serving model trained on all January: {metrics.get('serving_model_size_mb', float('nan')):,.0f} MB")
print(f"Registered {MODEL_NAME} version {tags.get('registered_version')} without an alias; "
      f"champion currently -> {ss.alias_version(MODEL_NAME, 'champion')}")

## 8. Conclusion

In [ ]:
rejected = ranked[ranked["status"] != "selected"]
print(f"Selected for serving: {selected_name} ({params.get('selected_n_estimators')} trees, "
      f"max_depth={params.get('selected_max_depth')}, min_samples_leaf={params.get('selected_min_samples_leaf')}, "
      f"max_features={params.get('selected_max_features')})")
print(f"  validation RMSE {selected['val_rmse']:.4f} / MAE {selected['val_mae']:.4f} "
      f"(best candidate {ranked['val_rmse'].min():.4f})")
print(f"  February holdout RMSE {metrics['holdout_rmse']:.4f} / MAE {metrics['holdout_mae']:.4f}")
print(f"  {selected['size_mb']:,.0f} MB, peak RAM {selected['peak_ram_mb']:,.0f} MB, "
      f"load {selected['load_s']:.1f} s, p95 {selected['latency_p95_ms']:.2f} ms")
print(f"Rejected: {len(rejected)} candidates - "
      f"{rejected['reason'].str.contains('RMSE').sum()} on accuracy, "
      f"{rejected['reason'].str.contains('budget').sum()} on size, "
      f"{rejected['reason'].str.startswith('eligible').sum()} lost on RAM/latency")

- The selection follows the rule in section 2 exactly; section 6 gives the reason for every rejected
  candidate.
- Validation numbers compare candidates; the **February** numbers are the only unbiased estimate for
  the deployed configuration, because they were computed once, after the choice.
- To serve the new version: set the `champion` alias to it by hand in the MLflow UI, then call the
  API's `/reload`. The previous champion stays available for rollback.